### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\ashu2\AppData\Local\Temp\ipykernel_14672\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\ashu2\OneDrive\Documents\LnGr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_all_pdfs(pdf_directory):
    all_docs = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing : {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['soucre_file'] = pdf_file.name
                doc.metadata['file_typr'] = 'pdf'

            all_docs.extend(documents)
            print(f"✅ Loaded {len(documents)} pages.")

        except Exception as e:
            print(f"Error : {e}")

    print(f"Total documents loaded : {len(all_docs)}")
    return all_docs

all_pdf_docs = process_all_pdfs("../data")


Found 4 PDF files to process

Processing : just-text.pdf
✅ Loaded 1 pages.

Processing : Opportunities and Challenges Related to AI in Education.pdf
✅ Loaded 9 pages.

Processing : text-and-images.pdf
✅ Loaded 2 pages.

Processing : text-and-table.pdf
✅ Loaded 2 pages.
Total documents loaded : 14


In [3]:
# Text Splitting 
def split_documents(documents, chunk_size = 1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"\nExample Chunk: ")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [4]:
chunks = split_documents(all_pdf_docs)

Split 14 documents into 69 chunks.

Example Chunk: 
Content: SamplePDFFileforTesting&Practice
Loremipsumdolorsitamet,consecteturadipiscingelit.Fusceidligulaacquamaliquetvenenatis.Utatligulatincidunt,sollicitudinpurusnec,blanditmetus.Sedauctorvenenatispurus,etve...
Metadata: {'producer': 'Skia/PDF m117 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Sample PDF FIle: Just Text', 'source': '..\\data\\pdfs\\just-text.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'soucre_file': 'just-text.pdf', 'file_typr': 'pdf'}


In [5]:
# Embeddings 
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load The Sentence Transformer model"""

        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded Successfullly. Embedding Dimension: {self.model.get_embedding_dimension()}")  
        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Returns : Numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not Loaded")

        print(f"Generating embeddings for {len(texts)} texts.")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## Initialize the Embedding Manager
embedding_manager = EmbeddingManager()
embedding_manager


Loading Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3285.72it/s]


Model Loaded Successfullly. Embedding Dimension: 384


In [7]:
# Vector Store
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        collection_name: Name of the ChromaDB collection
        persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None 
        self._intialize_store()

    def _intialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or Create collection
            self.collection = self.client.get_or_create_collection(
                name= self.collection_name,
                metadata={"description":"PDF Documents embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing Documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error : {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add docs amd embeds to the vector store
        documents: List of langchain docs
        embeddings: Corresponding embeddings for the docs
        """
        if len(documents) != len(embeddings):
            raise ValueError("No. of Docs must macth no. of embeds.")
        print(f"Adding {len(documents)} documents to vector store...")

        #prepare data for chromadb
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embed) in enumerate(zip(documents, embeddings)):
             #generate unique ID 
             doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
             ids.append(doc_id)

             #prepare metadata
             metadata = dict(doc.metadata)
             metadata['doc_index'] = i
             metadata['context_length'] = len(doc.page_content)
             metadatas.append(metadata)

             #Document content
             documents_text.append(doc.page_content)

             #Embedding
             embeddings_list.append(embed.tolist())

        #Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list, 
                metadatas = metadatas, 
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} docs to vector store.")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing Documents in collection: 207


In [8]:
# Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

#generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

#store in vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 69 texts.


Batches: 100%|██████████| 3/3 [00:01<00:00,  1.67it/s]


Generated embeddings with shape: (69, 384)
Adding 69 documents to vector store...
Successfully added 69 docs to vector store.
Total documents in collection: 276


In [9]:
class RAG_Retreiver:
    """Handles Query-based retreival from vector store"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retreiver
        Args:
            vector_store: Vector Store containing document embeddings
            embedding_manager: Manager for generating query embedding
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        Args: 
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        #Generate Query Embeddings 
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #Search in Vector Store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results = top_k,
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadata = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadata, distances)):
                    #Convert distance to similiarity score(ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document, 
                            'metadata':metadata, 
                            'similarity_score':similarity_score,
                            'distance':distance, 
                            'rank':i + 1
                        })

                    print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found") 

            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_ret = RAG_Retreiver(vectorstore, embedding_manager)
rag_ret

In [10]:
rag_ret.retrieve("What is AI in Education?")

Retrieving documents for query: 'What is AI in Education?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 34.24it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)
Retrieved 4 documents (after filtering)
Retrieved 5 documents (after filtering)


[{'id': 'doc_6f729b03_5',
  'content': 'The capability of computers to have a thought process and \nbehavioural patterns exactly like that of  humans, is called \nartificial intelligence (AI), and also, those efforts that are \nmade to  build such computerised systems that can imitate \nhuman thinking and behaviour. Given its potential to b e a \npotent element in guaranteeing economic development, \nartificial intelligence is a  perfect fit  to be set as the \nfundamental element for the 5th Industrial Revolution [2]. \nThe field of teaching and learning is being drastically \naltered by artificial intelligence (AI). Artificial Intelligence \n(AI) in education refers to the use of sophisticated computer \nsystems to echo the human intelligence and implement them \nin areas such a s learning, solving  a problem , and making  \ndecisions, thereby completing tasks that have historically \nrequired human cognition. This entails utilizing technology to \naddress the complex social, emotion

### Integrating Vector DB Context Pipeline with LLM Output

In [ ]:
# Simple RAG Pipeline with Groq LLM
from langchain_groq import ChatGroq
from pydantic import SecretStr
import os 
from dotenv import load_dotenv
load_dotenv() 

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY is not set.")

llm = ChatGroq(
    api_key=SecretStr(groq_api_key),
    model="llama-3.3-70b-versatile", 
    temperature=0.1, 
    max_tokens=1024
    )

#Simple RAG function: retrieve content + generate response
def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return " No relevant context found to answer the question"

    # Generate the answer using GROQ LLM
    prompt = f"""Use the following Context to answer the question concisely 
        Context:
        {context}
        Question: {query}
        Answer:"""

    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [18]:
answer = rag_simple("What are the Opportunities related to AI in Education", rag_ret, llm)
print(answer)

Retrieving documents for query: 'What are the Opportunities related to AI in Education'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 99.53it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)


The opportunities related to AI in education include:

1. **Developing new skills**: Focusing on abilities that AI cannot imitate, such as empathy, creativity, critical thinking, and teamwork.
2. **New pedagogies**: Creating new functions, methodologies, and pedagogies for a new learning and teaching environment.
3. **Enhanced monitoring**: Using AI to monitor AI-generated content and maintain academic integrity.
4. **Reconsidering university roles**: Universities can reassess their roles and adapt to the changing educational landscape.
5. **Human-computer collaboration**: The possibility of the human brain "crossbreeding" with computers, leading to new learning and teaching environments.


### Enchanced RAG Pipeline 

In [19]:
def rag_adv(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG Pipeline with extra features:
    Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer':'No relevant context found','sources':[], 'confidence':0.0, 'context':''}

    #Prepare Context and Sources
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results ]
    confidence = max([doc['similarity_score'] for doc in results])

    #Generate answer
    prompt = f"""Use the following context to answer the question concisely. \nContext: \n{context}\n\nQuestion: {query}\n\nAnswer"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer':response.content,
        'sources': sources, 
        'confidence':confidence
    }
    if return_context:
        output['context'] = context
    return output

result = rag_adv("What are the Opportunities related to AI in Education", rag_ret, llm, top_k = 3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What are the Opportunities related to AI in Education'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 86.46it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)


Answer: The opportunities related to AI in education include:

1. **Developing new skills**: Focusing on abilities that AI cannot imitate, such as empathy, creativity, critical thinking, and teamwork.
2. **New pedagogies**: Forcing educators to develop new functions, methodologies, and pedagogies for a new learning and teaching environment.
3. **Enhanced monitoring**: Allowing teachers and professors to monitor AI-generated content to maintain academic integrity.
4. **Reconsidering university roles**: Encouraging universities to reassess their roles and adapt to the changing educational landscape.

These opportunities enable educators to leverage AI to create a more effective and innovative learning environment.
Sources: [{'source': '..\\data\\pdfs\\Opportunities and Challenges Related to AI in Education.pdf', 'page': 5, 'score': 0.36240965127944946, 'preview': 'The emphasis shifts to developing abilities that AI cannot \nimitate, such as empathy, creativity, critical thinking, and \nt